# Oop

Split from `01_classes_and_oop.ipynb` to keep this topic self-contained.

**Navigation:** [Previous: Classes](./01_classes.ipynb) · [Topic overview](./01_classes_and_oop.ipynb)


# Module 13: Classes and Object-Oriented Programming

## Building Bioinformatics Tools with OOP

---

### Learning Objectives
- Understand classes, objects, and the `__init__` constructor
- Master `__str__`, `__repr__`, `__eq__`, `__lt__` dunder methods
- Implement inheritance, polymorphism, and abstract classes
- Use properties, class methods, and static methods
- Apply dataclasses for concise data-holding classes
- Build a mini BioPython-like `Seq` object from scratch

---

---

[← Previous: Regular Expressions for Bioinformatics](../../Tier_1_Python_for_Bioinformatics/12_Regular_Expressions/01_regular_expressions.ipynb) | [Next: Module 14: Decorators and Context Managers →](../../Tier_1_Python_for_Bioinformatics/14_Decorators_and_Context_Managers/01_decorators_and_context_managers.ipynb)

---

## 1. Why OOP in Bioinformatics?

Bioinformatics deals with complex, structured data: genes have names, sequences,
chromosomal locations, and behaviors (translation, complement). Representing a gene
as a dictionary works for toy examples, but quickly becomes unwieldy:

```python
gene = {"name": "BRCA1", "sequence": "ATG..."}
# How do you translate? Where is gc_content()? No structure, no safety.
```

Classes bundle **data** (attributes) and **behavior** (methods) into a single unit:

```
CLASS = Blueprint                 OBJECT = Actual instance
+-----------------+               +-----------------+
|     Gene        |               |  brca1          |
+-----------------+    creates    +-----------------+
| name            |  -------->>  | name="BRCA1"    |
| sequence        |               | sequence="ATG.."|
+-----------------+               +-----------------+
| translate()     |               | translate()     |
| gc_content()    |               | gc_content()    |
+-----------------+               +-----------------+
```

## 2. Defining a Class: `__init__`, Attributes, Methods

In [ ]:
class Gene:
    """Represents a gene with its DNA sequence and metadata."""

    def __init__(self, name, sequence, chromosome=None):
        """
        Initialize a Gene object.

        Parameters:
            name: Gene symbol (e.g. "BRCA1")
            sequence: DNA sequence string
            chromosome: Chromosome number (optional)
        """
        self.name = name
        self.sequence = sequence.upper()
        self.chromosome = chromosome

    def length(self):
        """Return sequence length in base pairs."""
        return len(self.sequence)

    def gc_content(self):
        """Calculate GC content as a percentage."""
        gc = self.sequence.count('G') + self.sequence.count('C')
        return (gc / len(self.sequence)) * 100


# Create Gene objects (instances)
brca1 = Gene("BRCA1", "ATGGATTTCGATCGATCGTAGC", chromosome=17)
tp53 = Gene("TP53", "ATGGAGGAGCCGCAGTCAGATC", chromosome=17)

print(f"Gene: {brca1.name}")
print(f"Length: {brca1.length()} bp")
print(f"GC Content: {brca1.gc_content():.1f}%")
print(f"Chromosome: {brca1.chromosome}")

### How `__init__` Works

When you write `brca1 = Gene("BRCA1", "ATG...")`, Python:

1. Creates a new, empty `Gene` object
2. Calls `__init__(new_object, "BRCA1", "ATG...")` -- `self` refers to the new object
3. Returns the initialized object

`self` is not a keyword -- it is a convention. The first parameter of every instance method
is the object itself.

## 3. `__str__` and `__repr__`: Controlling How Objects Print

In [ ]:
class Sequence:
    """A biological sequence with proper string representations."""

    def __init__(self, seq_id, sequence, seq_type="DNA"):
        self.seq_id = seq_id
        self.sequence = sequence.upper()
        self.seq_type = seq_type

    def __str__(self):
        """Human-readable: called by print() and str()."""
        if len(self.sequence) > 20:
            display = f"{self.sequence[:10]}...{self.sequence[-10:]}"
        else:
            display = self.sequence
        return f"{self.seq_id} ({self.seq_type}, {len(self.sequence)} bp): {display}"

    def __repr__(self):
        """Developer-facing: unambiguous, could recreate the object."""
        return f"Sequence('{self.seq_id}', '{self.sequence}', seq_type='{self.seq_type}')"


seq = Sequence("BRCA1_exon1", "ATGGATTTCGATCGATCGTAGCGATCGATCGATCG")

# __str__ is called by print()
print(seq)

# __repr__ is called by repr() and shown in interactive prompts
print(repr(seq))

**Rule of thumb:**
- `__str__` -- for the user (readable output)
- `__repr__` -- for the developer (should ideally let you recreate the object)
- If only one is defined, use `__repr__` -- it serves as fallback for `__str__` too

## 4. More Dunder Methods: `__len__`, `__eq__`, `__lt__`

In [ ]:
class DNASequence:
    """DNA sequence with rich comparison support."""

    VALID_BASES = set('ATGCN')

    def __init__(self, sequence, name="unnamed"):
        self.name = name
        sequence = sequence.upper()
        invalid = set(sequence) - self.VALID_BASES
        if invalid:
            raise ValueError(f"Invalid nucleotides: {invalid}")
        self.sequence = sequence

    def __len__(self):
        """Allow len(dna_seq) to work."""
        return len(self.sequence)

    def __eq__(self, other):
        """Two DNASequence objects are equal if their sequences match."""
        if not isinstance(other, DNASequence):
            return NotImplemented
        return self.sequence == other.sequence

    def __lt__(self, other):
        """Compare by length (useful for sorting)."""
        if not isinstance(other, DNASequence):
            return NotImplemented
        return len(self.sequence) < len(other.sequence)

    def __repr__(self):
        return f"DNASequence('{self.sequence[:20]}...', name='{self.name}')"


# Equality
s1 = DNASequence("ATGCATGC", "seq1")
s2 = DNASequence("ATGCATGC", "seq2")  # different name, same sequence
s3 = DNASequence("ATGCATGCATGC", "seq3")

print(f"s1 == s2: {s1 == s2}")   # True -- same sequence
print(f"s1 == s3: {s1 == s3}")   # False
print(f"s1 < s3:  {s1 < s3}")    # True -- shorter

# Sorting works because __lt__ is defined
sequences = [s3, s1, DNASequence("AT", "tiny")]
for s in sorted(sequences):
    print(f"  {s.name}: {len(s)} bp")

## 5. Inheritance: Building a Sequence Hierarchy

Inheritance lets you create specialized classes from a general base class.
DNA, RNA, and protein sequences share common features (they all have a sequence
string, a name, a length) but differ in their specific operations.

```
               BioSequence (Parent)
              /        |         \
         DNA          RNA       Protein
      gc_content()  to_dna()   mol_weight()
      complement()             
      transcribe()
```

In [ ]:
class BioSequence:
    """Base class for all biological sequences."""

    def __init__(self, sequence, name="unnamed"):
        self.sequence = sequence.upper()
        self.name = name

    def __len__(self):
        return len(self.sequence)

    def __str__(self):
        return f">{self.name}\n{self.sequence}"

    def __contains__(self, motif):
        """Allow 'ATG' in seq syntax."""
        return motif.upper() in self.sequence

    def composition(self):
        """Return character frequency dict."""
        return {char: self.sequence.count(char) for char in sorted(set(self.sequence))}


class DNA(BioSequence):
    """DNA sequence with DNA-specific methods."""

    COMPLEMENT_MAP = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}

    def gc_content(self):
        gc = self.sequence.count('G') + self.sequence.count('C')
        return (gc / len(self.sequence)) * 100

    def complement(self):
        """Return the complementary DNA strand."""
        comp = ''.join(self.COMPLEMENT_MAP[nt] for nt in self.sequence)
        return DNA(comp, name=f"{self.name}_comp")

    def reverse_complement(self):
        """Return the reverse complement."""
        comp = ''.join(self.COMPLEMENT_MAP[nt] for nt in reversed(self.sequence))
        return DNA(comp, name=f"{self.name}_revcomp")

    def transcribe(self):
        """Transcribe DNA to RNA (T -> U)."""
        return RNA(self.sequence.replace('T', 'U'), name=f"{self.name}_rna")


class RNA(BioSequence):
    """RNA sequence."""

    def to_dna(self):
        return DNA(self.sequence.replace('U', 'T'), name=f"{self.name}_dna")


class Protein(BioSequence):
    """Protein sequence with mass calculation."""

    # Average residue weights in Daltons
    AA_WEIGHTS = {
        'A': 89, 'R': 174, 'N': 132, 'D': 133, 'C': 121,
        'E': 147, 'Q': 146, 'G': 75, 'H': 155, 'I': 131,
        'L': 131, 'K': 146, 'M': 149, 'F': 165, 'P': 115,
        'S': 105, 'T': 119, 'W': 204, 'Y': 181, 'V': 117
    }

    def molecular_weight(self):
        """Calculate molecular weight in Daltons (subtract water per peptide bond)."""
        weight = sum(self.AA_WEIGHTS.get(aa, 110) for aa in self.sequence)
        weight -= (len(self.sequence) - 1) * 18  # water loss
        return weight

In [ ]:
# Test the hierarchy
dna = DNA("ATGCGATCGATCGTAGCGATCG", name="test_gene")

print(f"DNA: {dna.sequence}")
print(f"Length: {len(dna)} bp")
print(f"GC Content: {dna.gc_content():.1f}%")
print(f"Composition: {dna.composition()}")
print(f"Contains ATG: {'ATG' in dna}")

rev_comp = dna.reverse_complement()
print(f"\nReverse complement: {rev_comp.sequence}")

rna = dna.transcribe()
print(f"RNA: {rna.sequence}")

# isinstance checks
print(f"\ndna is DNA: {isinstance(dna, DNA)}")
print(f"dna is BioSequence: {isinstance(dna, BioSequence)}")
print(f"rna is DNA: {isinstance(rna, DNA)}")

## 6. Abstract Base Classes

Sometimes you want to define an interface that subclasses **must** implement.
Python's `abc` module enforces this at instantiation time.

In [ ]:
from abc import ABC, abstractmethod


class SequenceAnalyzer(ABC):
    """Abstract base class for sequence analyzers."""

    def __init__(self, sequence):
        self.sequence = sequence.upper()

    @abstractmethod
    def validate(self):
        """Check that the sequence is valid. Must be implemented."""
        ...

    @abstractmethod
    def summary(self):
        """Return a summary dict. Must be implemented."""
        ...


# This will raise TypeError -- cannot instantiate abstract class
try:
    analyzer = SequenceAnalyzer("ATGC")
except TypeError as e:
    print(f"Cannot instantiate abstract class: {e}")


class DNAAnalyzer(SequenceAnalyzer):
    """Concrete implementation for DNA."""

    def validate(self):
        invalid = set(self.sequence) - set('ATGCN')
        if invalid:
            raise ValueError(f"Invalid DNA bases: {invalid}")
        return True

    def summary(self):
        gc = (self.sequence.count('G') + self.sequence.count('C')) / len(self.sequence) * 100
        return {
            'length': len(self.sequence),
            'gc_content': round(gc, 2),
            'composition': {b: self.sequence.count(b) for b in 'ATGC'}
        }


analyzer = DNAAnalyzer("ATGCGATCGATCG")
analyzer.validate()
print(analyzer.summary())

## 7. Properties: Controlled Attribute Access

Properties let you add validation or computation behind attribute access,
without changing how the caller uses the object.

In [ ]:
class Gene:
    """Gene with validated attributes via properties."""

    VALID_STRANDS = {'+', '-'}

    def __init__(self, name, sequence, strand='+'):
        self.name = name
        self.sequence = sequence  # goes through the setter
        self.strand = strand      # goes through the setter

    @property
    def sequence(self):
        return self._sequence

    @sequence.setter
    def sequence(self, value):
        value = value.upper()
        invalid = set(value) - set('ATGCN')
        if invalid:
            raise ValueError(f"Invalid nucleotides in sequence: {invalid}")
        self._sequence = value

    @property
    def strand(self):
        return self._strand

    @strand.setter
    def strand(self, value):
        if value not in self.VALID_STRANDS:
            raise ValueError(f"Strand must be '+' or '-', got '{value}'")
        self._strand = value

    @property
    def gc_content(self):
        """Read-only computed property."""
        gc = self._sequence.count('G') + self._sequence.count('C')
        return (gc / len(self._sequence)) * 100


gene = Gene("TP53", "ATGGAGGAGCCGCAGTCAGATC", strand='+')
print(f"{gene.name}: GC = {gene.gc_content:.1f}%")

# Validation in action
try:
    gene.strand = 'X'
except ValueError as e:
    print(f"Caught: {e}")

try:
    gene.sequence = "ATGXYZ"
except ValueError as e:
    print(f"Caught: {e}")

## 8. Class Methods and Static Methods

In [ ]:
class FastaRecord:
    """A FASTA record with alternative constructors."""

    def __init__(self, seq_id, sequence, description=""):
        self.seq_id = seq_id
        self.sequence = sequence.upper()
        self.description = description

    @classmethod
    def from_fasta_string(cls, fasta_text):
        """Alternative constructor: parse a FASTA-formatted string."""
        lines = fasta_text.strip().split('\n')
        header = lines[0]
        if not header.startswith('>'):
            raise ValueError("FASTA header must start with '>'")
        parts = header[1:].split(None, 1)
        seq_id = parts[0]
        description = parts[1] if len(parts) > 1 else ""
        sequence = ''.join(lines[1:])
        return cls(seq_id, sequence, description)

    @staticmethod
    def is_valid_dna(sequence):
        """Check if a string is valid DNA (no instance needed)."""
        return set(sequence.upper()) <= set('ATGCN')

    def __str__(self):
        desc = f" {self.description}" if self.description else ""
        return f">{self.seq_id}{desc}\n{self.sequence}"


# Normal constructor
rec1 = FastaRecord("seq1", "ATGCGATCG", "test sequence")

# Class method constructor -- parses FASTA text
fasta_text = ">BRCA1 breast cancer gene\nATGGATTTCGATCGATCGTAGC"
rec2 = FastaRecord.from_fasta_string(fasta_text)
print(rec2)

# Static method -- no instance required
print(f"\nIs 'ATGC' valid DNA? {FastaRecord.is_valid_dna('ATGC')}")
print(f"Is 'ATGXZ' valid DNA? {FastaRecord.is_valid_dna('ATGXZ')}")

**When to use each:**

| Type | Receives | Use case |
|------|----------|----------|
| Regular method | `self` (instance) | Operates on instance data |
| `@classmethod` | `cls` (class) | Alternative constructors, factory methods |
| `@staticmethod` | nothing | Utility functions logically related to the class |

## 9. Dataclasses: Less Boilerplate

For classes that mainly hold data, `dataclasses` auto-generate `__init__`, `__repr__`,
`__eq__`, and optionally `__lt__` (with `order=True`).

In [ ]:
from dataclasses import dataclass, field


@dataclass(order=True)
class GeneAnnotation:
    """A gene annotation entry (e.g., from a GFF file)."""

    # Fields used for sorting (order=True uses field order)
    chromosome: str
    start: int
    end: int

    # Fields excluded from comparison
    name: str = field(compare=False, default="")
    strand: str = field(compare=False, default='+')
    gene_type: str = field(compare=False, default="protein_coding")

    @property
    def length(self):
        return self.end - self.start

    def overlaps(self, other):
        """Check if two annotations overlap on the same chromosome."""
        if self.chromosome != other.chromosome:
            return False
        return self.start < other.end and other.start < self.end


# Dataclass auto-generates __init__, __repr__, __eq__, __lt__
genes = [
    GeneAnnotation("chr17", 43044295, 43170245, name="BRCA1", strand='-'),
    GeneAnnotation("chr17", 7668402, 7687538, name="TP53", strand='-'),
    GeneAnnotation("chr7", 55019017, 55211628, name="EGFR", strand='+'),
    GeneAnnotation("chr17", 43050000, 43060000, name="NBR2", strand='+'),
]

# Auto __repr__
print(genes[0])

# Sorting works (by chromosome, then start, then end)
print("\nSorted by genomic position:")
for g in sorted(genes):
    print(f"  {g.name}: {g.chromosome}:{g.start}-{g.end} ({g.length:,} bp)")

# Overlap detection
brca1 = genes[0]
nbr2 = genes[3]
print(f"\nBRCA1 overlaps NBR2? {brca1.overlaps(nbr2)}")

## 10. Polymorphism in Action

Polymorphism means different classes respond to the same method call
in their own way. This is powerful for building generic pipelines.

In [ ]:
class BioSequence:
    """Base with a summary method that subclasses override."""

    def __init__(self, sequence, name="unnamed"):
        self.sequence = sequence.upper()
        self.name = name

    def summary(self):
        return f"{self.name}: {len(self.sequence)} residues"


class DNASeq(BioSequence):
    def summary(self):
        gc = (self.sequence.count('G') + self.sequence.count('C')) / len(self.sequence) * 100
        return f"{self.name}: {len(self.sequence)} bp, GC={gc:.1f}%"


class ProteinSeq(BioSequence):
    def summary(self):
        mw = len(self.sequence) * 110 / 1000  # rough kDa
        return f"{self.name}: {len(self.sequence)} aa, ~{mw:.1f} kDa"


# Generic function that works on any BioSequence
def print_report(sequences):
    """Works with any object that has a .summary() method."""
    for seq in sequences:
        print(f"  {seq.summary()}")


records = [
    DNASeq("ATGCGATCGATCGTAGCGATCG", "BRCA1_fragment"),
    DNASeq("GGCCAATTGGCCAATTGGCC", "GC_rich_region"),
    ProteinSeq("MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH", "HBA1"),
]

print("Sequence Report:")
print_report(records)

## 11. Complete Bioinformatics Class: Gene with Translation and ORF Finding

In [ ]:
class Gene:
    """A comprehensive Gene class for bioinformatics analysis."""

    GENETIC_CODE = {
        'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
        'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
        'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
        'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
        'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
        'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
        'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
        'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
        'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
        'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
        'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
        'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
        'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
        'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
        'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
        'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
    }

    COMPLEMENT_MAP = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}

    def __init__(self, name, sequence, chromosome=None, start=None, strand='+'):
        self.name = name
        self.sequence = sequence.upper()
        self.chromosome = chromosome
        self.start = start
        self.strand = strand

    def __str__(self):
        loc = f"chr{self.chromosome}:{self.start}" if self.chromosome else "unknown"
        return f"{self.name} ({loc}, {len(self)} bp, {self.strand} strand)"

    def __repr__(self):
        return f"Gene('{self.name}', '{self.sequence[:20]}...', chromosome={self.chromosome})"

    def __len__(self):
        return len(self.sequence)

    def gc_content(self):
        gc = self.sequence.count('G') + self.sequence.count('C')
        return (gc / len(self.sequence)) * 100

    def complement(self):
        """Return the complementary sequence string."""
        return ''.join(self.COMPLEMENT_MAP[nt] for nt in self.sequence)

    def reverse_complement(self):
        """Return the reverse complement string."""
        return ''.join(self.COMPLEMENT_MAP[nt] for nt in reversed(self.sequence))

    def get_codons(self, frame=0):
        """Extract codons from a specified reading frame (0, 1, or 2)."""
        seq = self.sequence[frame:]
        return [seq[i:i+3] for i in range(0, len(seq) - len(seq) % 3, 3)]

    def translate(self, frame=0):
        """Translate DNA to protein (stops at first stop codon)."""
        codons = self.get_codons(frame)
        protein = []
        for codon in codons:
            aa = self.GENETIC_CODE.get(codon, 'X')
            if aa == '*':
                break
            protein.append(aa)
        return ''.join(protein)

    def find_orfs(self, min_length=100):
        """Find all open reading frames (ATG to stop) >= min_length bp."""
        orfs = []
        for frame in range(3):
            codons = self.get_codons(frame)
            in_orf = False
            orf_start = 0
            for i, codon in enumerate(codons):
                if codon == 'ATG' and not in_orf:
                    in_orf = True
                    orf_start = frame + i * 3
                elif codon in ('TAA', 'TAG', 'TGA') and in_orf:
                    orf_end = frame + (i + 1) * 3
                    length = orf_end - orf_start
                    if length >= min_length:
                        orfs.append({
                            'start': orf_start,
                            'end': orf_end,
                            'length': length,
                            'frame': frame + 1,
                        })
                    in_orf = False
        return orfs

In [ ]:
# Test the complete Gene class
brca1 = Gene(
    name="BRCA1",
    sequence=(
        "ATGGATTTATCTGCTCTTCGCGTTGAAGAAGTACAAAATGTCATTAATGCTATGCAGAAA"
        "ATCTTAGAGTGTCCCATCTGTCTGGAGTTGATCAAGGAACCTGTCTCCACAAAGTGTGAC"
        "CACATATTTTGCAAATTTTGCATGCTGAAACTTCTCAACCAGAAGAAAGGGCCTTCACAG"
        "TGTCCTTTATGTAAGAATGATATAACCAAAAGGAGCCTACAAGAAAGTACGAGATTTAGT"
        "CAACTTGTTGAAGAGCTATTGAAAATCATTTGTGCTTTTCAGCTTGACACAGGTTTGGAG"
        "TATGCAAACAGCTATAATTTTGCAAAAAAGGAAAATAACTCTCCTGAACATCTAAAAGATG"
        "AAGTTTCTATCATCCAAAGTATGGGCTACAGAAACCGTGCCAAAAGACTTCTACAGAGTGA"
        "ACCCGAAAATCCTTCCTTGTAG"
    ),
    chromosome=17,
    start=43044295,
    strand='-'
)

print(brca1)
print(f"Length: {len(brca1)} bp")
print(f"GC Content: {brca1.gc_content():.1f}%")

protein = brca1.translate()
print(f"\nProtein ({len(protein)} aa): {protein[:60]}...")

orfs = brca1.find_orfs(min_length=30)
print(f"\nORFs found (min 30 bp): {len(orfs)}")
for orf in orfs:
    print(f"  Frame {orf['frame']}: {orf['start']}-{orf['end']} ({orf['length']} bp)")

## 12. ProteinStructure Class with Hydrophobicity Analysis

In [ ]:
class ProteinStructure:
    """Protein with structural and physicochemical analysis."""

    KD_HYDROPHOBICITY = {
        'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
        'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
        'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
        'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2
    }

    def __init__(self, accession, sequence, description="", organism=None):
        self.accession = accession
        self.sequence = sequence.upper()
        self.description = description
        self.organism = organism

    def __str__(self):
        org = f" [{self.organism}]" if self.organism else ""
        return f">{self.accession} {self.description}{org}\n{self.sequence}"

    def __len__(self):
        return len(self.sequence)

    def amino_acid_composition(self):
        """Return amino acid percentages."""
        total = len(self.sequence)
        return {aa: self.sequence.count(aa) / total * 100
                for aa in sorted(set(self.sequence))}

    def hydrophobicity_profile(self, window=7):
        """Kyte-Doolittle hydrophobicity with sliding window."""
        half = window // 2
        profile = []
        for i in range(len(self.sequence)):
            start = max(0, i - half)
            end = min(len(self.sequence), i + half + 1)
            window_seq = self.sequence[start:end]
            avg = sum(self.KD_HYDROPHOBICITY.get(aa, 0) for aa in window_seq) / len(window_seq)
            profile.append(avg)
        return profile

    def find_motif(self, motif):
        """Find all positions of a motif in the sequence."""
        positions = []
        start = 0
        motif = motif.upper()
        while True:
            pos = self.sequence.find(motif, start)
            if pos == -1:
                break
            positions.append(pos)
            start = pos + 1
        return positions


insulin = ProteinStructure(
    accession="P01308",
    sequence="MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN",
    description="Insulin precursor",
    organism="Homo sapiens"
)

print(insulin)
print(f"\nLength: {len(insulin)} aa")

# Find cysteines -- critical for disulfide bonds in insulin
cys_positions = insulin.find_motif('C')
print(f"Cysteine positions: {cys_positions}")
print(f"Number of cysteines: {len(cys_positions)} (insulin has 6 Cys for 3 disulfide bonds)")

# Hydrophobicity profile (first 20 values)
profile = insulin.hydrophobicity_profile(window=9)
print(f"\nHydrophobicity (first 20 residues): {[round(v, 2) for v in profile[:20]]}")

## 13. Design Challenge: A Mini BioPython-Like Seq Object

Let us build a `Seq` class inspired by BioPython's `Seq`, supporting
slicing, concatenation, and codon iteration.

In [ ]:
class Seq:
    """A mini BioPython-like Seq object.

    Supports:
    - Slicing: seq[3:10]
    - Concatenation: seq1 + seq2
    - Iteration: for base in seq
    - len(), str(), repr()
    - Complement, reverse complement, transcription, translation
    """

    COMPLEMENT_MAP = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}

    CODON_TABLE = {
        'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
        'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
        'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
        'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
        'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
        'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
        'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
        'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
        'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
        'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
        'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
        'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
        'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
        'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
        'TAC':'Y', 'TAT':'Y', 'TAA':'*', 'TAG':'*',
        'TGC':'C', 'TGT':'C', 'TGA':'*', 'TGG':'W',
    }

    def __init__(self, data):
        self._data = str(data).upper()

    # --- String representations ---
    def __str__(self):
        return self._data

    def __repr__(self):
        if len(self._data) > 60:
            return f"Seq('{self._data[:54]}...{self._data[-3:]}')"
        return f"Seq('{self._data}')"

    # --- Container protocol ---
    def __len__(self):
        return len(self._data)

    def __getitem__(self, index):
        """Support indexing and slicing: seq[0], seq[3:10]."""
        result = self._data[index]
        if isinstance(index, slice):
            return Seq(result)
        return result

    def __iter__(self):
        return iter(self._data)

    def __contains__(self, item):
        return str(item).upper() in self._data

    # --- Arithmetic ---
    def __add__(self, other):
        """Concatenate two Seq objects."""
        if isinstance(other, Seq):
            return Seq(self._data + other._data)
        return Seq(self._data + str(other).upper())

    # --- Comparison ---
    def __eq__(self, other):
        if isinstance(other, Seq):
            return self._data == other._data
        return self._data == str(other).upper()

    def __hash__(self):
        return hash(self._data)

    # --- Biological methods ---
    def complement(self):
        """Return the complement as a Seq."""
        return Seq(''.join(self.COMPLEMENT_MAP.get(b, 'N') for b in self._data))

    def reverse_complement(self):
        """Return the reverse complement as a Seq."""
        return Seq(''.join(self.COMPLEMENT_MAP.get(b, 'N') for b in reversed(self._data)))

    def transcribe(self):
        """DNA -> RNA (T -> U)."""
        return Seq(self._data.replace('T', 'U'))

    def back_transcribe(self):
        """RNA -> DNA (U -> T)."""
        return Seq(self._data.replace('U', 'T'))

    def translate(self):
        """Translate DNA to protein (stops at first stop codon)."""
        protein = []
        for i in range(0, len(self._data) - 2, 3):
            codon = self._data[i:i+3]
            aa = self.CODON_TABLE.get(codon, 'X')
            if aa == '*':
                break
            protein.append(aa)
        return Seq(''.join(protein))

    def gc_content(self):
        """Return GC fraction (0-1)."""
        gc = self._data.count('G') + self._data.count('C')
        return gc / len(self._data)

    def count_overlap(self, sub):
        """Count overlapping occurrences of a subsequence."""
        sub = str(sub).upper()
        count = 0
        start = 0
        while True:
            pos = self._data.find(sub, start)
            if pos == -1:
                break
            count += 1
            start = pos + 1
        return count

In [ ]:
# Demonstrate the Seq object
seq = Seq("ATGGATTTATCTGCTCTTCGCGTTGAAG")
print(f"Seq: {seq}")
print(f"Length: {len(seq)}")
print(f"GC content: {seq.gc_content():.2%}")

# Slicing returns a new Seq
codon = seq[0:3]
print(f"\nFirst codon: {codon} (type: {type(codon).__name__})")

# Biological operations
print(f"Complement: {seq.complement()}")
print(f"Rev comp:   {seq.reverse_complement()}")
print(f"RNA:        {seq.transcribe()}")
print(f"Protein:    {seq.translate()}")

# Concatenation
combined = seq + Seq("TTTAAACCC")
print(f"\nCombined: {combined} ({len(combined)} bp)")

# Membership test
print(f"Contains 'ATG': {'ATG' in seq}")

# Iteration
print(f"First 6 bases: {[b for b in seq[:6]]}")

## 14. Composition Over Inheritance: Restriction Enzyme Example

Sometimes it is better to compose objects rather than inherit.

In [ ]:
class RestrictionEnzyme:
    """A restriction enzyme that can find and cut DNA."""

    def __init__(self, name, recognition_site, cut_position):
        self.name = name
        self.recognition_site = recognition_site.upper()
        self.cut_position = cut_position

    def __repr__(self):
        site = self.recognition_site
        cut_display = site[:self.cut_position] + '|' + site[self.cut_position:]
        return f"RestrictionEnzyme('{self.name}', '{cut_display}')"

    def find_sites(self, sequence):
        """Return positions of all recognition sites."""
        seq = sequence.upper()
        positions = []
        start = 0
        while True:
            pos = seq.find(self.recognition_site, start)
            if pos == -1:
                break
            positions.append(pos)
            start = pos + 1
        return positions

    def cut(self, sequence):
        """Cut the sequence at all recognition sites."""
        sites = self.find_sites(sequence)
        if not sites:
            return [sequence]
        # Calculate actual cut positions
        cut_positions = [s + self.cut_position for s in sites]
        fragments = []
        prev = 0
        for cp in cut_positions:
            fragments.append(sequence[prev:cp])
            prev = cp
        fragments.append(sequence[prev:])
        return fragments


# Common restriction enzymes
ecori = RestrictionEnzyme("EcoRI", "GAATTC", 1)   # G|AATTC
hindiii = RestrictionEnzyme("HindIII", "AAGCTT", 1) # A|AGCTT

plasmid = "ATGCGAATTCGGGAATTCTAGAAGCTTCCG"

print(f"{ecori}")
sites = ecori.find_sites(plasmid)
print(f"EcoRI sites in plasmid: {sites}")

fragments = ecori.cut(plasmid)
print(f"Fragments after EcoRI digest:")
for i, frag in enumerate(fragments, 1):
    print(f"  Fragment {i}: {frag} ({len(frag)} bp)")

---

## Exercises

### Exercise 1: Extend the Seq Class

Add the following methods to the `Seq` class:
- `find(sub)` -- return the first position of a subsequence, or -1
- `__mul__(n)` -- repeat the sequence n times (e.g., `seq * 3`)
- `codons()` -- return a list of 3-letter codon strings

In [ ]:
# Exercise 1: Your solution
# Extend the Seq class with find(), __mul__, and codons() methods.

# Test:
# s = Seq("ATGGCTTAG")
# print(s.find("GCT"))  # 3
# print(s * 2)           # ATGGCTTAGATGGCTTAG
# print(s.codons())      # ['ATG', 'GCT', 'TAG']

### Exercise 2: Gene Hierarchy with RNA Translation

Create an `mRNA` class that inherits from `BioSequence` (section 5) and has:
- `__init__` that validates only A, U, G, C are present
- A `translate()` method that converts RNA codons to amino acids
- A class method `from_dna(dna_sequence)` that transcribes and returns an mRNA

In [ ]:
# Exercise 2: Your solution

# Test:
# rna = mRNA("AUGGCUUAG", name="test")
# print(rna.translate())         # "MA"
# rna2 = mRNA.from_dna("ATGGCTTAG")
# print(rna2.sequence)           # "AUGGCUUAG"

### Exercise 3: Sortable ProteinRecord Dataclass

Create a `ProteinRecord` dataclass with fields: `accession`, `name`, `sequence`, `organism`, `length`.
- `length` should be computed from `sequence` in `__post_init__`
- Enable sorting by length
- Add a method `has_signal_peptide()` that returns True if the first 20 amino acids
  contain more than 60% hydrophobic residues (A, I, L, M, F, V, W)

In [ ]:
# Exercise 3: Your solution

# Test:
# proteins = [
#     ProteinRecord("P01308", "Insulin", "MALWMRLLPLLALLALWGPD" + "A" * 80, "Human"),
#     ProteinRecord("P68871", "Hemoglobin B", "MVHLTPEEKSAVTALWGKV" + "G" * 30, "Human"),
# ]
# for p in sorted(proteins):
#     print(f"{p.name}: {p.length} aa, signal peptide: {p.has_signal_peptide()}")

### Exercise 4: Genome Annotation System

Using the `GeneAnnotation` dataclass from section 9, write a function
`find_overlapping_genes(annotations)` that returns a list of tuples
`(gene_a, gene_b)` for all pairs of genes that overlap on the same chromosome.

In [ ]:
# Exercise 4: Your solution

# Test:
# annotations = [
#     GeneAnnotation("chr1", 100, 500, name="GeneA"),
#     GeneAnnotation("chr1", 400, 800, name="GeneB"),
#     GeneAnnotation("chr1", 900, 1200, name="GeneC"),
#     GeneAnnotation("chr2", 100, 500, name="GeneD"),
# ]
# overlaps = find_overlapping_genes(annotations)
# for a, b in overlaps:
#     print(f"{a.name} overlaps {b.name}")  # GeneA overlaps GeneB

### Exercise 5: Build a MultipleSequenceAlignment Class

Create a class `MSA` that stores aligned sequences (all same length) and provides:
- `__init__(self, sequences)` where sequences is a dict `{name: aligned_sequence}`
- Validation that all sequences have the same length
- `conservation(position)` -- return the fraction of sequences that have the most common residue at that position
- `consensus()` -- return a consensus string (most common residue at each position)
- `__len__` -- return the alignment length

In [ ]:
# Exercise 5: Your solution

# Test:
# alignment = MSA({
#     "Human":  "MVLSPADKTNVKAAWGKVGA",
#     "Chimp":  "MVLSPADKTNVKAAWGKVGA",
#     "Gorilla":"MVLSPADKTNVKAAWGKVGA",
#     "Mouse":  "MVLSGEDKSNIKAAWGKIGA",
# })
# print(f"Alignment length: {len(alignment)}")
# print(f"Consensus: {alignment.consensus()}")
# print(f"Conservation at pos 5: {alignment.conservation(5):.0%}")

---

## Summary

```
OOP CONCEPTS COVERED
+----------------------------------------------------------+
|  BASICS:                                                  |
|  class, __init__, self, attributes, methods               |
|                                                           |
|  DUNDER METHODS:                                          |
|  __str__, __repr__, __len__, __eq__, __lt__               |
|  __getitem__, __contains__, __add__, __iter__             |
|                                                           |
|  INHERITANCE & POLYMORPHISM:                              |
|  super(), method overriding, isinstance(), ABC            |
|                                                           |
|  ADVANCED PATTERNS:                                       |
|  @property, @classmethod, @staticmethod, @dataclass       |
|                                                           |
|  BIOINFORMATICS CLASSES:                                  |
|  Seq (mini BioPython), Gene, ProteinStructure,            |
|  RestrictionEnzyme, GeneAnnotation, BioSequence hierarchy |
+----------------------------------------------------------+
```

---

[← Previous: Regular Expressions for Bioinformatics](../../Tier_1_Python_for_Bioinformatics/12_Regular_Expressions/01_regular_expressions.ipynb) | [Next: Module 14: Decorators and Context Managers →](../../Tier_1_Python_for_Bioinformatics/14_Decorators_and_Context_Managers/01_decorators_and_context_managers.ipynb)